## 0. Config


In [1]:
import os, subprocess, textwrap, shutil, random, zipfile
from pathlib import Path

SEQ        = 'zavod70'

DRIVE_DIR  = '/content/drive/MyDrive/fvtest'
ZIP        = f'{DRIVE_DIR}/{SEQ}.zip'
SAVE       = f'{DRIVE_DIR}/result'

CONDA      = '/opt/conda'
VIPE_DIR   = '/content/vipe'
GS_DIR     = '/content/gaussian-splatting'
DATA       = f'/content/data/{SEQ}'
VIPE_OUT   = '/content/vipe_results'
SCENE_ROOT = '/content/colmap'
SCENE      = f'{SCENE_ROOT}/{SEQ}'
MODEL      = f'/content/output/{SEQ}'

ITERS      = 30000

os.environ['HF_HOME'] = '/content/hf_cache'


def bash(script):
    """Run bash, stream the output, raise on failure."""
    prelude = ('set -eo pipefail\n'
               f'source {CONDA}/etc/profile.d/conda.sh 2>/dev/null || true\n')
    p = subprocess.Popen(['bash', '-c', prelude + textwrap.dedent(script)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=os.environ.copy())
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'exit code {p.returncode}')


In [2]:
import torch

VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
major, minor = torch.cuda.get_device_capability()

PIPELINE    = 'default' if VRAM > 20 else 'no_vda'
DATA_DEVICE = 'cuda' if VRAM > 30 else 'cpu'

os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'

print(torch.cuda.get_device_name(0), f'{VRAM:.0f} GB, sm_{major}{minor}')
print(f'pipeline={PIPELINE}  data_device={DATA_DEVICE}')


NVIDIA A100-SXM4-40GB 39 GB, sm_80
pipeline=default  data_device=cuda


## 1. Install


In [3]:
%%time
# System packages and conda
bash(f'''
    apt-get update -qq
    apt-get install -y -qq git wget ffmpeg build-essential ninja-build libgl1 libglib2.0-0
    if [ ! -d {CONDA} ]; then
        wget -qO /tmp/mf.sh \\
          https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
        bash /tmp/mf.sh -b -p {CONDA}
    fi
    {CONDA}/bin/conda --version
''')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../wget_1.21.2-2ubuntu1.4_amd64.deb ...
Unpacking wget (1.21.2-2ubuntu1.4) over (1.21.2-2ubuntu1.1) ...
Selecting previously unselected package ninja-build.
Preparing to unpa

In [4]:
%%time
# Vipe install
bash(f'''
    [ -d {VIPE_DIR} ] || git clone --depth 1 https://github.com/nv-tlabs/vipe.git {VIPE_DIR}
    cd {VIPE_DIR}

    # The NVIDIA conda channel drops connections regularly - be patient with it.
    conda config --set remote_max_retries 10
    conda config --set remote_connect_timeout_secs 30
    conda config --set remote_read_timeout_secs 120

    NVCC={CONDA}/envs/cu128/bin/nvcc
    if [ -d {CONDA}/envs/cu128 ] && [ ! -x $NVCC ]; then
        conda env remove -n cu128 -y
    fi
    for attempt in 1 2 3; do
        [ -x $NVCC ] && break
        echo "=== conda env create, attempt $attempt ==="
        conda env create -f envs/cu128.yml || sleep 15
    done
    [ -x $NVCC ] || {{ echo 'cu128 env incomplete after 3 attempts'; exit 1; }}

    conda activate cu128
    export CUDA_HOME=$CONDA_PREFIX
    export MAX_JOBS=$(nproc)

    # Build against uv's own CPython, not the system one. Debian splits its Python
    # headers across /usr/include/python3.10 and /usr/include/x86_64-linux-gnu, and
    # the conda compiler does not search the second path - pyconfig.h then fails.
    # uv's standalone build keeps its headers in one place.
    export UV_PYTHON_PREFERENCE=only-managed
    uv python install 3.10
    rm -rf .venv

    uv sync --python 3.10
    uv run python -c "import vipe; print('vipe ok')"
''')


Cloning into '/content/vipe'...
=== conda env create, attempt 1 ===
Retrieving notices: - \ done
Channels:
 - nvidia/label/cuda-12.8.0
 - conda-forge
Platform: linux-64
Solving environment: / done

libcublas-12.8.3.14  | 460.0 MB  |            |   0% 

libcusparse-12.5.7.5 | 165.1 MB  |            |   0% 


libcusolver-11.7.2.5 | 157.0 MB  |            |   0% 



gcc_impl_linux-64-13 | 67.1 MB   |            |   0% 




cuda-nvrtc-12.8.61   | 63.2 MB   |            |   0% 





sysroot_linux-64-2.3 | 38.9 MB   |            |   0% 






libnvjitlink-12.8.61 | 28.8 MB   |            |   0% 







cuda-nvcc-tools-12.8 | 24.5 MB   |            |   0% 








cuda-nvvm-tools-12.8 | 23.5 MB   |            |   0% 









cuda-nvvm-impl-12.8. | 20.8 MB   |            |   0% 










libstdcxx-devel_linu | 18.1 MB   |            |   0% 











uv-0.12.5            | 16.9 MB   |            |   0% 












gxx_impl_linux-64-13 | 13.4 MB   |            |   0% 













cud

In [5]:
%%time
# Gaussian Splatting install
bash(f'''
    [ -d {GS_DIR} ] || git clone --recursive --depth 1 \\
        https://github.com/graphdeco-inria/gaussian-splatting.git {GS_DIR}
    cd {GS_DIR}
    pip install -q plyfile
    export CUDA_HOME=/usr/local/cuda
    pip install -q submodules/diff-gaussian-rasterization
    pip install -q submodules/simple-knn
    pip install -q submodules/fused-ssim
''')

import diff_gaussian_rasterization, simple_knn, fused_ssim


Cloning into '/content/gaussian-splatting'...
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
Cloning into '/content/gaussian-splatting/submodules/diff-gaussian-rasterization'...
Cloning into '/content/gaussian-splatting/submodules/fused-ssim'...
Cloning into '/content/gaussian-splatting/submodules/simple-knn'...
From https://gitlab.inria.fr/sibr/sibr_core
 * branch            d8856f60c5384cc1975439193bb627d77d917d77 -> FETCH_HEAD
Submodule path 'S

## 2. Data

Unpacks the zip onto the local disk.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

shutil.rmtree(DATA, ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/data')

frames = sorted(Path(DATA).glob('*.jpg'))
assert frames, f'no frames in {DATA} - does the zip contain a {SEQ}/ folder?'
print(f'{len(frames)} frames in {DATA}')


Mounted at /content/drive
126 frames in /content/data/zavod70


## 3. ViPE inference

In [7]:
%%time
bash(f'''
    cd {VIPE_DIR}
    conda activate cu128
    HYDRA_FULL_ERROR=1 MPLBACKEND=agg uv run python run.py \\
        pipeline={PIPELINE} \\
        streams=frame_dir_stream \\
        streams.base_path={DATA} \\
        pipeline.output.path={VIPE_OUT} \\
        pipeline.output.save_artifacts=true \\
        pipeline.output.save_slam_map=true \\
        pipeline.output.save_viz=false
''')


2026-08-18 19:02:34,278 - vipe - INFO - Processing zavod70 (1 / 1)
2026-08-18 19:02:34,282 - vipe.utils.model_cache - INFO - Building and caching model 'geocalib/pinhole'
Downloading: "https://github.com/cvg/GeoCalib/releases/download/v1.0/geocalib-pinhole.tar" to /root/.cache/torch/hub/geocalib/pinhole.tar

  0%|          | 0.00/111M [00:00<?, ?B/s]
  9%|▉         | 10.1M/111M [00:00<00:02, 41.1MB/s]
 18%|█▊        | 20.1M/111M [00:00<00:02, 42.3MB/s]
 27%|██▋       | 30.1M/111M [00:00<00:02, 40.0MB/s]
 36%|███▌      | 40.1M/111M [00:01<00:01, 40.6MB/s]
 45%|████▌     | 50.1M/111M [00:01<00:01, 41.3MB/s]
 54%|█████▍    | 60.1M/111M [00:01<00:02, 25.5MB/s]
 63%|██████▎   | 70.1M/111M [00:02<00:01, 29.4MB/s]
 72%|███████▏  | 80.1M/111M [00:02<00:00, 32.3MB/s]
 81%|████████▏ | 90.1M/111M [00:02<00:00, 35.2MB/s]
 90%|█████████ | 100M/111M [00:03<00:00, 36.4MB/s] 
 99%|█████████▉| 110M/111M [00:03<00:00, 38.2MB/s]
100%|██████████| 111M/111M [00:03<00:00, 35.8MB/s]

Caching: 100%|██████████

## 4. COLMAP scene


In [8]:
%%time
bash(f'''
    cd {VIPE_DIR}
    conda activate cu128
    uv run python scripts/vipe_to_colmap.py {VIPE_OUT} \\
        --sequence {SEQ} \\
        --output {SCENE_ROOT} \\
        --use_slam_map
''')


2026-08-18 19:05:09,110 - __main__ - INFO - Converting ViPE results from /content/vipe_results (zavod70) to COLMAP format at /content/colmap/zavod70
2026-08-18 19:05:09,110 - __main__ - INFO - Extracting frames from /content/vipe_results/rgb/zavod70.mp4
2026-08-18 19:05:09,363 - __main__ - INFO - Extracted 0 frames
2026-08-18 19:05:09,525 - __main__ - INFO - Extracted 30 frames
2026-08-18 19:05:09,673 - __main__ - INFO - Extracted 60 frames
2026-08-18 19:05:09,819 - __main__ - INFO - Extracted 90 frames
2026-08-18 19:05:09,964 - __main__ - INFO - Extracted 120 frames
2026-08-18 19:05:09,991 - __main__ - INFO - Extracted 125 frames to /content/colmap/zavod70/images
2026-08-18 19:05:09,993 - __main__ - INFO - Written cameras.txt with intrinsics: fx=478.60, fy=478.60, cx=320.00, cy=240.00
2026-08-18 19:05:10,017 - __main__ - INFO - Written images.txt with 126 images
2026-08-18 19:05:12,399 - __main__ - INFO - COLMAP conversion completed successfully!
2026-08-18 19:05:12,399 - __main__ - I

In [9]:
# The converter writes the model into the scene root; 3DGS reads it from sparse/0.
# The cloud is also unbounded, and 3DGS makes one Gaussian per point.
MAX_POINTS = 3_000_000

sparse = Path(SCENE) / 'sparse' / '0'
sparse.mkdir(parents=True, exist_ok=True)
for name in ('cameras.txt', 'images.txt', 'points3D.txt'):
    f = Path(SCENE) / name
    if f.exists():
        shutil.move(str(f), str(sparse / name))

# vipe_to_colmap writes NAME as 'images/frame_000046.jpg', and the Inria loader
# joins that onto the images folder - giving images/images/... Strip the prefix.
img_txt = sparse / 'images.txt'
out, fixed = [], 0
for line in img_txt.read_text().splitlines():
    parts = line.split()
    # image lines are 'ID QW QX QY QZ TX TY TZ CAM_ID NAME'; the empty points2D
    # lines between them carry no data but must be preserved
    if not line.startswith('#') and len(parts) == 10 and '/' in parts[-1]:
        parts[-1] = parts[-1].rsplit('/', 1)[-1]
        line = ' '.join(parts)
        fixed += 1
    out.append(line)
img_txt.write_text('\n'.join(out) + '\n')
print(f'stripped path prefix from {fixed} image names')

pts = sparse / 'points3D.txt'
lines  = pts.read_text().splitlines()
header = [l for l in lines if l.startswith('#')]
points = [l for l in lines if l and not l.startswith('#')]
print(f'{len(points):,} points')

if len(points) > MAX_POINTS:
    random.seed(0)
    sample = random.sample(points, MAX_POINTS)
    points = [' '.join([str(i)] + l.split()[1:]) for i, l in enumerate(sample, start=1)]
    header = [h for h in header if not h.startswith('# Number of points')]
    header.append(f'# Number of points: {len(points)}')
    pts.write_text('\n'.join(header + points) + '\n')
    print(f'capped to {len(points):,}')

print(f'scene ready: {SCENE}')
print(f"images: {len(list((Path(SCENE) / 'images').glob('*.jpg')))}")


stripped path prefix from 126 image names
314,499 points
scene ready: /content/colmap/zavod70
images: 126


## 5. Train GS

In [10]:
%%time
bash(f'''
    cd {GS_DIR}
    python train.py \\
        -s {SCENE} \\
        -m {MODEL} \\
        -r 1 \\
        --iterations {ITERS} \\
        --data_device {DATA_DEVICE}
''')


Streaming output truncated to the last 5000 lines.
Training progress: 100%|██████████| 30000/30000 [26:39<00:00, 18.76it/s, Loss=0.0358697, Depth Loss=0.0000000]

[ITER 7000] Evaluating train: L1 0.05769392549991608 PSNR 21.335465621948245 [18/08 19:08:24]

[ITER 7000] Saving Gaussians [18/08 19:08:24]

[ITER 30000] Evaluating train: L1 0.02358561158180237 PSNR 27.809754180908204 [18/08 19:32:10]

[ITER 30000] Saving Gaussians [18/08 19:32:10]

Training complete. [18/08 19:33:25]
CPU times: user 2.51 s, sys: 348 ms, total: 2.86 s
Wall time: 28min 14s


## 6. Render

In [11]:
%%time
bash(f'''
    cd {GS_DIR}
    python render.py -m {MODEL} --iteration {ITERS} --skip_test
''')

RENDERS = f'{MODEL}/train/ours_{ITERS}/renders'
VIDEO   = f'/content/{SEQ}.mp4'
bash(f'''
    ffmpeg -y -loglevel error -framerate 20 -i {RENDERS}/%05d.png \\
        -c:v libx264 -pix_fmt yuv420p -crf 18 {VIDEO}
''')


Looking for config file in /content/output/zavod70/cfg_args
Config file found: /content/output/zavod70/cfg_args
Rendering /content/output/zavod70
Loading trained model at iteration 30000 [18/08 19:33:31]

Reading camera 1/126
Reading camera 2/126
Reading camera 3/126
Reading camera 4/126
Reading camera 5/126
Reading camera 6/126
Reading camera 7/126
Reading camera 8/126
Reading camera 9/126
Reading camera 10/126
Reading camera 11/126
Reading camera 12/126
Reading camera 13/126
Reading camera 14/126
Reading camera 15/126
Reading camera 16/126
Reading camera 17/126
Reading camera 18/126
Reading camera 19/126
Reading camera 20/126
Reading camera 21/126
Reading camera 22/126
Reading camera 23/126
Reading camera 24/126
Reading camera 25/126
Reading camera 26/126
Reading camera 27/126
Reading camera 28/126
Reading camera 29/126
Reading camera 30/126
Reading camera 31/126
Reading camera 32/126
Reading camera 33/126
Reading camera 34/126
Reading camera 35/126
Reading camera 36/126
Reading came

## 7. Save to Drive


In [12]:
Path(SAVE).mkdir(parents=True, exist_ok=True)

shutil.copy(VIDEO, SAVE)
shutil.copy(f'{MODEL}/point_cloud/iteration_{ITERS}/point_cloud.ply', SAVE)
print(SAVE)

/content/drive/MyDrive/fvtest/result
